# Baseline Comparison — Pipeline v2 (WINDOW_SIZE=60)

Same protocol as `module3_pipeline/baseline_comparison.ipynb`: Gaussian
whitened-distance score (zero learned parameters) and Isolation Forest,
compared against the v2 (window=60) VAE, all on the same PCA-whitened
features, same leak-free threshold discipline.

In [1]:
import numpy as np
import pickle, os
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score,
)

BASE     = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
WIN_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
MODEL_DIR = os.path.join(BASE, 'models_v2')

SETS = ['cc1_train', 'cc1_val', 'cc1_test', 'drift_cc2']
DRIFT_SETS = ['drift_cc2']

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

X, y = {}, {}
for name in SETS:
    X[name] = np.load(os.path.join(WIN_DIR, f'X_{name}.npy'))
    y[name] = np.load(os.path.join(WIN_DIR, f'y_{name}.npy'))
    print(f'  {name:10s}: {X[name].shape}  anomalies={int(y[name].sum()):,}')

meta = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
model.eval()

vae_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
print('\nVAE (v2, window=60) reference results loaded.')

  cc1_train : (152456, 48)  anomalies=0
  cc1_val   : (20763, 48)  anomalies=0
  cc1_test  : (43375, 48)  anomalies=256
  drift_cc2 : (76167, 48)  anomalies=1,080

VAE (v2, window=60) reference results loaded.


## Baseline A — Gaussian whitened-distance score

In [2]:
def gaussian_score(Xarr):
    return (Xarr ** 2).sum(axis=1)

scores_gaussian = {name: gaussian_score(X[name]) for name in SETS}
print('cc1_train whitening sanity check:')
print(f'  mean of per-component mean: {X["cc1_train"].mean():.4f}')
print(f'  mean of per-component std:  {X["cc1_train"].std():.4f}')

cc1_train whitening sanity check:
  mean of per-component mean: 0.0000
  mean of per-component std:  1.0000


## Baseline B — Isolation Forest

In [3]:
iso_forest = IsolationForest(n_estimators=100, contamination='auto', random_state=42, n_jobs=-1)
iso_forest.fit(X['cc1_train'])
scores_iso = {name: -iso_forest.decision_function(X[name]) for name in SETS}
print('Isolation Forest fit on cc1_train.')

Isolation Forest fit on cc1_train.


## Leak-free thresholds (from `cc1_val`) + full comparison

In [4]:
thresh_gaussian = float(np.percentile(scores_gaussian['cc1_val'], 99))
thresh_iso      = float(np.percentile(scores_iso['cc1_val'], 99))
print(f'Gaussian val_p99: {thresh_gaussian:.4f}   Isolation Forest val_p99: {thresh_iso:.4f}')

def evaluate(scores, y_true, threshold):
    auc_roc = roc_auc_score(y_true, scores)
    auc_pr  = average_precision_score(y_true, scores)
    pred = (scores > threshold).astype(int)
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)
    return {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'precision': p, 'recall': r, 'f1': f1}

results = {'gaussian': {}, 'isolation_forest': {}, 'vae': {}}
for name in ['cc1_test'] + DRIFT_SETS:
    results['gaussian'][name] = evaluate(scores_gaussian[name], y[name], thresh_gaussian)
    results['isolation_forest'][name] = evaluate(scores_iso[name], y[name], thresh_iso)
    results['vae'][name] = {
        'auc_roc': vae_eval['auc'][name]['auc_roc'],
        'auc_pr':  vae_eval['auc'][name]['auc_pr'],
        'precision': vae_eval['precision_recall'][name]['val_p99']['precision'],
        'recall':    vae_eval['precision_recall'][name]['val_p99']['recall'],
        'f1':        vae_eval['precision_recall'][name]['val_p99']['f1'],
    }

print(f'{"set":12s} {"method":18s} {"AUC-ROC":>9s} {"AUC-PR":>8s} {"precision":>10s} {"recall":>8s} {"F1":>7s}')
for name in ['cc1_test'] + DRIFT_SETS:
    for method in ['gaussian', 'isolation_forest', 'vae']:
        r = results[method][name]
        print(f'{name:12s} {method:18s} {r["auc_roc"]:9.4f} {r["auc_pr"]:8.4f} {r["precision"]:10.3f} {r["recall"]:8.3f} {r["f1"]:7.3f}')
    print()

Gaussian val_p99: 158.4432   Isolation Forest val_p99: 0.0248
set          method               AUC-ROC   AUC-PR  precision   recall      F1
cc1_test     gaussian              0.9439   0.7965      0.980    0.773   0.865
cc1_test     isolation_forest      0.8076   0.1159      0.131    0.176   0.150
cc1_test     vae                   0.9855   0.9178      0.986    0.848   0.912

drift_cc2    gaussian              0.9019   0.3151      0.118    0.815   0.206
drift_cc2    isolation_forest      0.8169   0.2528      0.175    0.386   0.241
drift_cc2    vae                   0.9129   0.3307      0.086    0.800   0.155



## Save

In [5]:
out_path = os.path.join(MODEL_DIR, 'baseline_comparison.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\models_v2\baseline_comparison.pkl
